# Flow Matching: learn a velocity field

**Flow Matching for Generative Modeling** · Lipman et al. · 2022

[Original paper](https://arxiv.org/abs/2210.02747)

**Learning goal:** Train a conditional velocity objective and sample by numerically solving an ODE.

## Before starting

Derivatives, MSE, Gaussian samples, and introductory VLA.

Allow roughly 40–60 minutes including exercises. CPU, PyTorch, and Matplotlib; no model or dataset downloads. This is a study-time estimate, not a runtime benchmark.

## Problem and idea

Flow matching learns a velocity field transporting a base distribution to data. Training can sample path points and target velocities without solving an ODE on every step; sampling integrates the learned field.

Action policies additionally condition the field on observations and instructions. We omit conditioning and study a one-dimensional distribution using independent noise/data pairs and linear interpolation with zero endpoint noise. This is a particular conditional path, not a global optimal-transport solution.

## Reproduction boundary

Simple linear conditional flow matching; no complete π0/SmolVLA, vision encoder, robot, or paper benchmark.

The code is an original educational implementation of mathematical ideas from the paper, not a copy of an author repository. Passing assertions verifies the stated implementation properties, not the paper’s results.

## Formula → tensors

`x_t = (1-t)*x0 + t*x1`; target velocity is `x1-x0`; `L = E[||v_theta(x_t,t) - (x1-x0)||²]`.

Each pair has constant velocity, but conditional paths overlap: the network predicts conditional mean velocity given x_t,t. Euler sampling uses `x ← x + dt*v_theta(x,t)` as t goes from 0 to 1. Reversing time requires consistently changing the sign/time convention.

In [ ]:
%matplotlib inline
import math
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(42)
plt.rcParams.update({'figure.figsize': (7, 3), 'axes.spines.top': False, 'axes.spines.right': False})
print('PyTorch:', torch.__version__, '| CPU | seed=42')

## Step 1

Verify endpoints and the path derivative with finite differences.

In [ ]:
x0 = torch.randn(32, 1); x1 = 2.0 + 0.5 * torch.randn(32, 1)
def path(t):
    return (1 - t) * x0 + t * x1
assert torch.equal(path(0), x0) and torch.equal(path(1), x1)
dt = 1e-3
derivative = (path(0.4 + dt) - path(0.4)) / dt
assert torch.allclose(derivative, x1 - x0, atol=3e-4)
print('endpoint means:', x0.mean().item(), x1.mean().item())

## Step 2

The model sees x_t and t, not both endpoints. Training targets and model inputs are different.

In [ ]:
velocity = nn.Sequential(nn.Linear(2, 48), nn.SiLU(), nn.Linear(48, 48), nn.SiLU(), nn.Linear(48, 1))
optimizer = torch.optim.Adam(velocity.parameters(), lr=0.003)
history = []
for step in range(300):
    base = torch.randn(128, 1); data = 2.0 + 0.5 * torch.randn(128, 1)
    t = torch.rand(128, 1); point = (1 - t) * base + t * data
    loss = F.mse_loss(velocity(torch.cat([point, t], -1)), data - base)
    optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step(); history.append(loss.item())
assert math.isfinite(history[-1])
plt.plot(history, color='#d73535', alpha=0.6); plt.xlabel('step'); plt.ylabel('velocity MSE'); plt.show()

## Step 3

Compare Euler step counts from identical initial noise. Differences reflect discretization and field approximation.

In [ ]:
initial = torch.randn(1000, 1)
def sample(steps):
    x = initial.clone()
    with torch.no_grad():
        for index in range(steps):
            t = torch.full_like(x, index / steps)
            x = x + velocity(torch.cat([x, t], -1)) / steps
    return x
coarse, fine = sample(5), sample(50)
assert torch.isfinite(fine).all()
print('Euler 5/50 means:', coarse.mean().item(), fine.mean().item(), 'target mean: 2')
print('fine std:', fine.std().item(), 'target std: 0.5')
plt.hist(fine.flatten(), bins=40, density=True, alpha=0.6, color='#d73535', label='generated')
plt.hist((2 + 0.5 * torch.randn(1000)).flatten(), bins=40, density=True, alpha=0.3, color='gray', label='target')
plt.legend(); plt.show()

## Reading the result

Training MSE need not reach zero: multiple pairs can share x_t,t but have different velocities. More sampling steps can reduce discretization error, not fix a wrong field. VLA also needs conditioning, action normalization, and dataset-specific masks.

## Independent experiments

1. Reverse the sampling velocity sign.
2. Compare 1, 5, 20, 100 steps with the same seed.
3. Add binary conditioning for target means -2 and +2.

Write a prediction first, then change one factor at a time. Reset the kernel and rerun all cells for a controlled comparison. Save a copy for your changes.

## Reasoning check — after experimenting

The wrong sign transports samples in the wrong direction. Compare identical initial samples. Supply conditioning labels at both training and sampling; otherwise the network mixes the tasks.

## What to record in your lab notes

- Hypothesis and changed parameter.
- Seed, data size, step budget, and precision.
- Before/after diagnostics, plot, and explanation of unexpected results.
- What this experiment **does not** establish.

Use “Open Studio workspace” above the lesson to continue in the related tools.